In [ ]:
"""
unet_model.py

Research-quality U-Net for Land Surface Temperature (LST) downscaling.

Inputs:
    - Sentinel-2 spectral bands
    - NDVI
    - NDBI
    - NASADEM elevation
    - ECOSTRESS LST (optional)

Output:
    - High-resolution LST prediction

Designed to be combined with a Physics-Informed Neural Network (PINN)
loss during training.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """
    Two consecutive Conv-BN-ReLU layers.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.double_conv = nn.Sequential(

`

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.double_conv(x)


class DownBlock(nn.Module):
    """
    Encoder block:
        DoubleConv
        ↓
        MaxPool
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(
            in_channels,
            out_channels
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled


class UpBlock(nn.Module):
    """
    Decoder block:
        Transposed Convolution
        ↓
        Concatenate Skip Connection
        ↓
        DoubleConv
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        if x.shape != skip.shape:

            x = F.interpolate(
                x,
                size=skip.shape[2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat(
            [skip, x],
            dim=1
        )

        x = self.conv(x)

        return x


class UNet(nn.Module):
    """
    U-Net architecture for LST regression.
    """

    def __init__(
        self,
        in_channels=10,
        out_channels=1
    ):

        super().__init__()

        # Encoder

        self.down1 = DownBlock(
            in_channels,
            64
        )

        self.down2 = DownBlock(
            64,
            128
        )

        self.down3 = DownBlock(
            128,
            256
        )

        self.down4 = DownBlock(
            256,
            512
        )

        # Bottleneck

        self.bottleneck = DoubleConv(
            512,
            1024
        )

        # Decoder

        self.up1 = UpBlock(
            1024,
            512
        )

        self.up2 = UpBlock(
            512,
            256
        )

        self.up3 = UpBlock(
            256,
            128
        )

        self.up4 = UpBlock(
            128,
            64
        )

        # Final Regression Layer

        self.final_conv = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

        self.initialize_weights()


    def forward(self, x):

        skip1, x = self.down1(x)

        skip2, x = self.down2(x)

        skip3, x = self.down3(x)

        skip4, x = self.down4(x)

        x = self.bottleneck(x)

        x = self.up1(x, skip4)

        x = self.up2(x, skip3)

        x = self.up3(x, skip2)

        x = self.up4(x, skip1)

        x = self.final_conv(x)

        return x


    def initialize_weights(self):

        for module in self.modules():

            if isinstance(module, nn.Conv2d):

                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu"
                )

            elif isinstance(module, nn.BatchNorm2d):

                nn.init.constant_(
                    module.weight,
                    1
                )

                nn.init.constant_(
                    module.bias,
                    0
                )


if __name__ == "__main__":

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = UNet(
        in_channels=10,
        out_channels=1
    ).to(device)

    x = torch.randn(
        2,
        10,
        256,
        256
    ).to(device)

    y = model(x)

    print("Input shape :", x.shape)
    print("Output shape:", y.shape)